In [20]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
from pathlib import Path
import sys
import os
import copy
import itertools
import random
import numpy as np
import pandas as pd
import torch
import importlib

BASE_DIR = Path("/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSSSM")
DATA_DIR = BASE_DIR / "data"
SRC_DIR = BASE_DIR / "src"
OUT_DIR = BASE_DIR / "outputs"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("SRC_DIR:", SRC_DIR)
print("OUT_DIR:", OUT_DIR)

print("Data files:")
print(list(DATA_DIR.iterdir()))

BASE_DIR: /content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSSSM
DATA_DIR: /content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSSSM/data
SRC_DIR: /content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSSSM/src
OUT_DIR: /content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSSSM/outputs
Data files:
[PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSSSM/data/X_val_4.npy'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSSSM/data/X_train_4.npy'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSSSM/data/dates_val_4.npy'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSSSM/data/dates_train_4.npy'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSSSM/data/miss_train_4.npy'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSSSM/data/miss_val_4.npy'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSSSM/data/obs_test_4.npy'), PosixPath('/content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSS

In [22]:
if "DSSSMCode" in sys.modules:
    del sys.modules["DSSSMCode"]

sys.path.insert(0, str(SRC_DIR))

import DSSSMCode
importlib.reload(DSSSMCode)

from DSSSMCode import (
    DSSSM,
    train_dsssm_model,
    extract_dsssm_regimes,
    regime_diagnostics,
    daily_segment_summary,
)

print("Imported from:", DSSSMCode.__file__)
print("Import successful.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Imported from: /content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSSSM/src/DSSSMCode.py
Import successful.
Using device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [23]:
X_train = np.load(DATA_DIR / "X_train_4.npy")
X_val   = np.load(DATA_DIR / "X_val_4.npy")
X_test  = np.load(DATA_DIR / "X_test_4.npy")

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (5522, 391, 19)
X_val: (1258, 391, 19)
X_test: (1453, 391, 19)


In [24]:
trainX = torch.tensor(np.transpose(X_train, (1, 0, 2)), dtype=torch.float32, device=device)
validX = torch.tensor(np.transpose(X_val,   (1, 0, 2)), dtype=torch.float32, device=device)
testX  = torch.tensor(np.transpose(X_test,  (1, 0, 2)), dtype=torch.float32, device=device)

trainY = trainX.clone()
validY = validX.clone()
testY  = testX.clone()

print("trainX:", trainX.shape, trainX.device)
print("validX:", validX.shape, validX.device)
print("testX:", testX.shape, testX.device)

trainX: torch.Size([391, 5522, 19]) cuda:0
validX: torch.Size([391, 1258, 19]) cuda:0
testX: torch.Size([391, 1453, 19]) cuda:0


In [18]:
search_space_dsssm = {
    "d_dim": [2],
    "z_dim": [2],
    "h_dim": [16],
    "lamda_b": [0.0, 1.0, 5.0, 10.0],
    "learning_rate": [1e-3],
}

candidate_configs_dsssm = [
    {
        "d_dim": d_dim,
        "z_dim": z_dim,
        "h_dim": h_dim,
        "lamda_b": lamda_b,
        "learning_rate": learning_rate,
    }
    for d_dim, z_dim, h_dim, lamda_b, learning_rate in itertools.product(
        search_space_dsssm["d_dim"],
        search_space_dsssm["z_dim"],
        search_space_dsssm["h_dim"],
        search_space_dsssm["lamda_b"],
        search_space_dsssm["learning_rate"],
    )
]

print("Number of DSSSM configs:", len(candidate_configs_dsssm))
candidate_configs_dsssm

Number of DSSSM configs: 4


[{'d_dim': 2, 'z_dim': 2, 'h_dim': 16, 'lamda_b': 0.0, 'learning_rate': 0.001},
 {'d_dim': 2, 'z_dim': 2, 'h_dim': 16, 'lamda_b': 1.0, 'learning_rate': 0.001},
 {'d_dim': 2, 'z_dim': 2, 'h_dim': 16, 'lamda_b': 5.0, 'learning_rate': 0.001},
 {'d_dim': 2,
  'z_dim': 2,
  'h_dim': 16,
  'lamda_b': 10.0,
  'learning_rate': 0.001}]

In [20]:
x_dim = X_train.shape[-1]
y_dim = X_train.shape[-1]

n_layers = 1
batch_size = 256
n_epochs = 40
patience = 15

results_dsssm = []
candidate_models_dsssm = {}
candidate_histories_dsssm = {}

for i, cfg in enumerate(candidate_configs_dsssm, start=1):

    print("=" * 100)
    print(f"Training DSSSM candidate {i}/{len(candidate_configs_dsssm)}")
    print(cfg)

    model = DSSSM(
        x_dim=x_dim,
        y_dim=y_dim,
        h_dim=cfg["h_dim"],
        z_dim=cfg["z_dim"],
        d_dim=cfg["d_dim"],
        n_layers=n_layers,
        device=device,
        bidirection=False,
        dataname="SPY",
        lamda_b=cfg["lamda_b"],
        lamda_entropy=0.0,
    ).to(device)

    print("Model device:", next(model.parameters()).device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg["learning_rate"],
    )

    history = train_dsssm_model(
        model=model,
        optimizer=optimizer,
        trainX=trainX,
        trainY=trainY,
        validX=validX,
        validY=validY,
        n_epochs=n_epochs,
        batch_size=batch_size,
        patience=patience,
        verbose=True,
    )

    states_val, probs_val = extract_dsssm_regimes(
        model,
        validX,
        validY,
    )

    diag_val = regime_diagnostics(
        states_val,
        n_states=cfg["d_dim"],
    )

    daily_summary, _ = daily_segment_summary(states_val)

    usage = diag_val["usage"]
    trans = diag_val["transition_matrix"]

    best_val_loss = float(np.min(history["val_loss"]))
    best_epoch = int(np.argmin(history["val_loss"]) + 1)

    row = {
        "d_dim": cfg["d_dim"],
        "z_dim": cfg["z_dim"],
        "h_dim": cfg["h_dim"],
        "lamda_b": cfg["lamda_b"],
        "learning_rate": cfg["learning_rate"],

        "best_epoch": best_epoch,
        "val_loss": best_val_loss,
        "val_nll": float(history["val_nll"][best_epoch - 1]),
        "val_kld_cat": float(history["val_kld_cat"][best_epoch - 1]),
        "val_balance": float(history["val_balance"][best_epoch - 1]),

        "min_usage": float(diag_val["min_usage"]),
        "max_usage": float(diag_val["max_usage"]),
        "eff_regimes": float(diag_val["effective_regimes"]),
        "switch_rate": float(diag_val["switch_rate"]),
        "avg_transition_diag": float(np.mean(np.diag(trans))),

        **daily_summary,
    }

    for k in range(cfg["d_dim"]):
        row[f"usage_{k}"] = float(usage[k])

    results_dsssm.append(row)

    key = (
        int(cfg["d_dim"]),
        int(cfg["z_dim"]),
        int(cfg["h_dim"]),
        float(cfg["lamda_b"]),
        float(cfg["learning_rate"]),
    )

    candidate_models_dsssm[key] = copy.deepcopy(model).cpu()
    candidate_histories_dsssm[key] = history

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_dsssm_df = pd.DataFrame(results_dsssm)
display(results_dsssm_df)

results_dsssm_df.to_csv(OUT_DIR / "dsssm_2state_grid_results.csv", index=False)

Training DSSSM candidate 1/4
{'d_dim': 2, 'z_dim': 2, 'h_dim': 16, 'lamda_b': 0.0, 'learning_rate': 0.001}
Model device: cuda:0
Epoch 001 | train_loss=27.721723 | val_loss=21.458277 | val_nll=20.976242 | val_kld_cat=0.056677 | val_balance=0.025559 | annealing=0.0100
Epoch 002 | train_loss=25.813480 | val_loss=20.257021 | val_nll=19.778402 | val_kld_cat=0.106509 | val_balance=0.050866 | annealing=0.0100
Epoch 003 | train_loss=23.330680 | val_loss=18.800776 | val_nll=18.243349 | val_kld_cat=0.202360 | val_balance=0.084455 | annealing=0.0100
Epoch 004 | train_loss=20.936039 | val_loss=17.039389 | val_nll=16.461750 | val_kld_cat=0.277286 | val_balance=0.177507 | annealing=0.0100
Epoch 005 | train_loss=18.591295 | val_loss=13.858733 | val_nll=13.311607 | val_kld_cat=0.325618 | val_balance=0.320720 | annealing=0.0100
Epoch 006 | train_loss=15.801103 | val_loss=9.715959 | val_nll=9.205214 | val_kld_cat=0.334070 | val_balance=0.370381 | annealing=0.0100
Epoch 007 | train_loss=12.652378 | val_l

,d_dim,z_dim,h_dim,lamda_b,learning_rate,best_epoch,val_loss,val_nll,val_kld_cat,val_balance,...,avg_transition_diag,median_segments,mean_segments,p75_segments,p90_segments,max_segments,one_state_share,multi_state_share,usage_0,usage_1
0,2,2,16,0.0,0.001,40,-19.063360,-19.195719,0.074183,0.328788,...,0.969692,5.0,5.136725,7.0,9.0,25,0.170111,0.829889,0.095774,0.904226
1,2,2,16,1.0,0.001,40,-18.130419,-18.583776,0.117364,0.040489,...,0.959617,15.0,15.310811,19.0,22.0,34,0.000000,1.000000,0.642275,0.357725
2,2,2,16,5.0,0.001,40,-19.290253,-19.909216,0.074505,0.050890,...,0.977941,8.0,8.548490,11.0,14.0,31,0.000000,1.000000,0.341099,0.658901
3,2,2,16,10.0,0.001,40,-17.635201,-19.433657,0.086873,0.025340,...,0.979060,8.0,8.781399,11.0,14.0,24,0.001590,0.998410,0.613388,0.386612


In [21]:
display(results_dsssm_df)

display(
    results_dsssm_df.sort_values(
        ["val_loss", "switch_rate"],
        ascending=[True, True]
    )
)

,d_dim,z_dim,h_dim,lamda_b,learning_rate,best_epoch,val_loss,val_nll,val_kld_cat,val_balance,...,avg_transition_diag,median_segments,mean_segments,p75_segments,p90_segments,max_segments,one_state_share,multi_state_share,usage_0,usage_1
0,2,2,16,0.0,0.001,40,-19.063360,-19.195719,0.074183,0.328788,...,0.969692,5.0,5.136725,7.0,9.0,25,0.170111,0.829889,0.095774,0.904226
1,2,2,16,1.0,0.001,40,-18.130419,-18.583776,0.117364,0.040489,...,0.959617,15.0,15.310811,19.0,22.0,34,0.000000,1.000000,0.642275,0.357725
2,2,2,16,5.0,0.001,40,-19.290253,-19.909216,0.074505,0.050890,...,0.977941,8.0,8.548490,11.0,14.0,31,0.000000,1.000000,0.341099,0.658901
3,2,2,16,10.0,0.001,40,-17.635201,-19.433657,0.086873,0.025340,...,0.979060,8.0,8.781399,11.0,14.0,24,0.001590,0.998410,0.613388,0.386612


,d_dim,z_dim,h_dim,lamda_b,learning_rate,best_epoch,val_loss,val_nll,val_kld_cat,val_balance,...,avg_transition_diag,median_segments,mean_segments,p75_segments,p90_segments,max_segments,one_state_share,multi_state_share,usage_0,usage_1
2,2,2,16,5.0,0.001,40,-19.290253,-19.909216,0.074505,0.050890,...,0.977941,8.0,8.548490,11.0,14.0,31,0.000000,1.000000,0.341099,0.658901
0,2,2,16,0.0,0.001,40,-19.063360,-19.195719,0.074183,0.328788,...,0.969692,5.0,5.136725,7.0,9.0,25,0.170111,0.829889,0.095774,0.904226
1,2,2,16,1.0,0.001,40,-18.130419,-18.583776,0.117364,0.040489,...,0.959617,15.0,15.310811,19.0,22.0,34,0.000000,1.000000,0.642275,0.357725
3,2,2,16,10.0,0.001,40,-17.635201,-19.433657,0.086873,0.025340,...,0.979060,8.0,8.781399,11.0,14.0,24,0.001590,0.998410,0.613388,0.386612


In [22]:
selection_dsssm_df = results_dsssm_df[
    (results_dsssm_df["min_usage"] >= 0.25) &
    (results_dsssm_df["eff_regimes"] >= 1.70) &
    (results_dsssm_df["switch_rate"] >= 0.003) &
    (results_dsssm_df["switch_rate"] <= 0.050) &
    (results_dsssm_df["median_segments"] >= 1) &
    (results_dsssm_df["median_segments"] <= 16) &
    (results_dsssm_df["p90_segments"] <= 28)
].copy()

if len(selection_dsssm_df) == 0:
    print("No strict admissible DSSSM models. Using relaxed criteria.")

    selection_dsssm_df = results_dsssm_df[
        (results_dsssm_df["min_usage"] >= 0.15) &
        (results_dsssm_df["eff_regimes"] >= 1.50) &
        (results_dsssm_df["switch_rate"] >= 0.001) &
        (results_dsssm_df["switch_rate"] <= 0.070) &
        (results_dsssm_df["median_segments"] <= 22) &
        (results_dsssm_df["p90_segments"] <= 34)
    ].copy()

selection_dsssm_df = selection_dsssm_df.sort_values(
    ["val_loss", "switch_rate"],
    ascending=[True, True]
)

display(selection_dsssm_df)

best_dsssm_row = selection_dsssm_df.iloc[0]

best_dsssm_key = (
    int(best_dsssm_row["d_dim"]),
    int(best_dsssm_row["z_dim"]),
    int(best_dsssm_row["h_dim"]),
    float(best_dsssm_row["lamda_b"]),
    float(best_dsssm_row["learning_rate"]),
)

print("Selected DSSSM key:", best_dsssm_key)
print(best_dsssm_row)

,d_dim,z_dim,h_dim,lamda_b,learning_rate,best_epoch,val_loss,val_nll,val_kld_cat,val_balance,...,avg_transition_diag,median_segments,mean_segments,p75_segments,p90_segments,max_segments,one_state_share,multi_state_share,usage_0,usage_1
2,2,2,16,5.0,0.001,40,-19.290253,-19.909216,0.074505,0.050890,...,0.977941,8.0,8.548490,11.0,14.0,31,0.00000,1.00000,0.341099,0.658901
1,2,2,16,1.0,0.001,40,-18.130419,-18.583776,0.117364,0.040489,...,0.959617,15.0,15.310811,19.0,22.0,34,0.00000,1.00000,0.642275,0.357725
3,2,2,16,10.0,0.001,40,-17.635201,-19.433657,0.086873,0.025340,...,0.979060,8.0,8.781399,11.0,14.0,24,0.00159,0.99841,0.613388,0.386612


Selected DSSSM key: (2, 2, 16, 5.0, 0.001)
d_dim                   2.000000
z_dim                   2.000000
h_dim                  16.000000
lamda_b                 5.000000
learning_rate           0.001000
best_epoch             40.000000
val_loss              -19.290253
val_nll               -19.909216
val_kld_cat             0.074505
val_balance             0.050890
min_usage               0.341099
max_usage               0.658901
eff_regimes             1.899825
switch_rate             0.019355
avg_transition_diag     0.977941
median_segments         8.000000
mean_segments           8.548490
p75_segments           11.000000
p90_segments           14.000000
max_segments           31.000000
one_state_share         0.000000
multi_state_share       1.000000
usage_0                 0.341099
usage_1                 0.658901
Name: 2, dtype: float64


In [23]:
final_dsssm_model = candidate_models_dsssm[best_dsssm_key].to(device)

print("Final model device:", next(final_dsssm_model.parameters()).device)

states_test_dsssm, probs_test_dsssm = extract_dsssm_regimes(
    final_dsssm_model,
    testX,
    testY,
)

diag_test = regime_diagnostics(
    states_test_dsssm,
    n_states=int(best_dsssm_row["d_dim"]),
)

test_daily_summary, test_daily_segments = daily_segment_summary(
    states_test_dsssm
)

print("DSSSM test usage:")
print(diag_test["usage"])

print("\nDSSSM effective regimes:")
print(diag_test["effective_regimes"])

print("\nDSSSM switch rate:")
print(diag_test["switch_rate"])

print("\nDSSSM transition matrix:")
print(diag_test["transition_matrix"])

print("\nDSSSM daily summary:")
print(test_daily_summary)

print("\nDSSSM daily segment distribution:")
print(test_daily_segments["segments"].value_counts().sort_index())

Final model device: cuda:0
DSSSM test usage:
[0.36347235 0.63652765]

DSSSM effective regimes:
1.9258933331813866

DSSSM switch rate:
0.02141634460973759

DSSSM transition matrix:
[[0.9684416  0.0315584 ]
 [0.01562786 0.98437214]]

DSSSM daily summary:
{'median_segments': 9.0, 'mean_segments': 9.35237439779766, 'p75_segments': 12.0, 'p90_segments': 15.0, 'max_segments': 28, 'one_state_share': 0.0, 'multi_state_share': 1.0}

DSSSM daily segment distribution:
segments
2      52
3      40
4     109
5      80
6     158
7     103
8     150
9     110
10    138
11     89
12    108
13     68
14     58
15     48
16     48
17     24
18     16
19     18
20     13
21      7
22      5
23      2
24      1
25      4
26      3
28      1
Name: count, dtype: int64


In [24]:
np.save(OUT_DIR / "dsssm_2state_test_states.npy", states_test_dsssm)
np.save(OUT_DIR / "dsssm_2state_test_probs.npy", probs_test_dsssm)

test_daily_segments.to_csv(
    OUT_DIR / "dsssm_2state_test_daily_segments.csv",
    index=False
)

torch.save(
    {
        "model_state_dict": final_dsssm_model.state_dict(),
        "best_key": best_dsssm_key,
        "best_row": best_dsssm_row.to_dict(),
        "x_dim": x_dim,
        "y_dim": y_dim,
        "n_layers": n_layers,
    },
    OUT_DIR / "dsssm_2state_selected_model.pt"
)

results_dsssm_df.to_csv(
    OUT_DIR / "dsssm_2state_grid_results.csv",
    index=False
)

selection_dsssm_df.to_csv(
    OUT_DIR / "dsssm_2state_selection_results.csv",
    index=False
)

print("Saved DSSSM outputs to:", OUT_DIR)

Saved DSSSM outputs to: /content/drive/MyDrive/Aueb_Thesis/Empirical_Study/DSSSM/outputs


disconnected

we compute now all the needed metrics for the DSSSM 2-state


In [30]:
import numpy as np
import pandas as pd
import torch

# Load saved test states/probs
states_test_dsssm_2 = np.load(OUT_DIR / "dsssm_2state_test_states.npy")
probs_test_dsssm_2 = np.load(OUT_DIR / "dsssm_2state_test_probs.npy")

print("states_test_dsssm_2:", states_test_dsssm_2.shape)
print("probs_test_dsssm_2:", probs_test_dsssm_2.shape)

# Load saved checkpoint
ckpt_2 = torch.load(
    OUT_DIR / "dsssm_2state_selected_model.pt",
    map_location=device
)

print("Saved best key:", ckpt_2["best_key"])
print("Saved best row:", ckpt_2["best_row"])

states_test_dsssm_2: (1453, 391)
probs_test_dsssm_2: (1453, 391, 2)
Saved best key: (2, 2, 16, 5.0, 0.001)
Saved best row: {'d_dim': 2.0, 'z_dim': 2.0, 'h_dim': 16.0, 'lamda_b': 5.0, 'learning_rate': 0.001, 'best_epoch': 40.0, 'val_loss': -19.290252685546875, 'val_nll': -19.909215927124023, 'val_kld_cat': 0.07450482249259949, 'val_balance': 0.05088980123400688, 'min_usage': 0.3410988090542777, 'max_usage': 0.6589011909457223, 'eff_regimes': 1.8998247075440389, 'switch_rate': 0.019355101708042884, 'avg_transition_diag': 0.9779414108420664, 'median_segments': 8.0, 'mean_segments': 8.548489666136724, 'p75_segments': 11.0, 'p90_segments': 14.0, 'max_segments': 31.0, 'one_state_share': 0.0, 'multi_state_share': 1.0, 'usage_0': 0.3410988090542777, 'usage_1': 0.6589011909457223}


In [31]:
best_dsssm_key = ckpt_2["best_key"]

d_dim, z_dim, h_dim, lamda_b, learning_rate = best_dsssm_key

final_dsssm_model = DSSSM(
    x_dim=ckpt_2["x_dim"],
    y_dim=ckpt_2["y_dim"],
    h_dim=h_dim,
    z_dim=z_dim,
    d_dim=d_dim,
    n_layers=ckpt_2["n_layers"],
    device=device,
    bidirection=False,
    dataname="SPY",
    lamda_b=lamda_b,
    lamda_entropy=0.0,
).to(device)

final_dsssm_model.load_state_dict(ckpt_2["model_state_dict"])
final_dsssm_model.eval()

print("2-state DSSSM model loaded.")
print("Model device:", next(final_dsssm_model.parameters()).device)

2-state DSSSM model loaded.
Model device: cuda:0


In [32]:
states_val_dsssm_2, probs_val_dsssm_2 = extract_dsssm_regimes(
    final_dsssm_model,
    validX,
    validY,
)

print("states_val_dsssm_2:", states_val_dsssm_2.shape)
print("probs_val_dsssm_2:", probs_val_dsssm_2.shape)

np.save(OUT_DIR / "dsssm_2state_val_states.npy", states_val_dsssm_2)
np.save(OUT_DIR / "dsssm_2state_val_probs.npy", probs_val_dsssm_2)

states_val_dsssm_2: (1258, 391)
probs_val_dsssm_2: (1258, 391, 2)


In [33]:
import pandas as pd
import numpy as np

panel_features = pd.read_parquet(DATA_DIR / "panel_features_4.parquet")

print(panel_features.shape)
print(panel_features.columns.tolist())

(3219103, 34)
['date', 'source_file', 'dataset_key', 'price_col', 'm_index', 'vwap', 'trade_flag', 'vwap_zero_filled', 'vwap_ffill', 'log_price_ffill', 'ret_1', 'sq_ret_1', 'abs_ret_1', 'y', 'ret_mean_15', 'ret_std_15', 'rv_15', 'momentum_15', 'trend_strength_15', 'ret_mean_30', 'ret_std_30', 'rv_30', 'momentum_30', 'trend_strength_30', 'ret_mean_60', 'ret_std_60', 'rv_60', 'momentum_60', 'trend_strength_60', 'ret_mean_120', 'ret_std_120', 'rv_120', 'momentum_120', 'trend_strength_120']


In [34]:
import numpy as np
import pandas as pd
import torch

# Load date arrays
dates_train = np.load(DATA_DIR / "dates_train_4.npy", allow_pickle=True)
dates_val   = np.load(DATA_DIR / "dates_val_4.npy", allow_pickle=True)
dates_test  = np.load(DATA_DIR / "dates_test_4.npy", allow_pickle=True)

dates_train = pd.to_datetime(dates_train)
dates_val   = pd.to_datetime(dates_val)
dates_test  = pd.to_datetime(dates_test)

print("dates:", len(dates_train), len(dates_val), len(dates_test))

# Prepare panel
panel_features["date"] = pd.to_datetime(panel_features["date"])

panel_features = (
    panel_features
    .sort_values(["date", "m_index"])
    .reset_index(drop=True)
)

# Create validation/test panels
val_panel = (
    panel_features[panel_features["date"].isin(dates_val)]
    .sort_values(["date", "m_index"])
    .reset_index(drop=True)
)

test_panel = (
    panel_features[panel_features["date"].isin(dates_test)]
    .sort_values(["date", "m_index"])
    .reset_index(drop=True)
)

print("val_panel:", val_panel.shape)
print("test_panel:", test_panel.shape)

print("Expected val rows:", X_val.shape[0] * X_val.shape[1])
print("Expected test rows:", X_test.shape[0] * X_test.shape[1])

assert len(val_panel) == X_val.shape[0] * X_val.shape[1]
assert len(test_panel) == X_test.shape[0] * X_test.shape[1]

print("Panels are aligned.")

dates: 5522 1258 1453
val_panel: (491878, 34)
test_panel: (568123, 34)
Expected val rows: 491878
Expected test rows: 568123
Panels are aligned.


In [35]:
econ_cols = [
    "ret_mean_30", "ret_mean_60", "ret_mean_120",
    "momentum_30", "momentum_60", "momentum_120",
    "trend_strength_30", "trend_strength_60", "trend_strength_120",
    "ret_std_60", "rv_60",
]

future_cols = ["future_ret_30", "future_ret_60", "future_ret_120"]


def add_future_returns(panel, return_col="ret_1", date_col="date", horizons=(30, 60, 120)):
    """
    Adds future cumulative returns within each day.
    future_ret_30 at minute t = sum of ret_1 from t+1 to t+30.
    """
    panel = panel.copy()

    for h in horizons:
        col = f"future_ret_{h}"

        if col in panel.columns:
            continue

        panel[col] = (
            panel
            .groupby(date_col)[return_col]
            .transform(
                lambda s: s.shift(-1)
                           .rolling(window=h, min_periods=1)
                           .sum()
                           .shift(-(h-1))
            )
        )

    return panel


def transition_matrix_from_states(states, n_states):
    """
    states shape: [n_days, T]
    """
    counts = np.zeros((n_states, n_states), dtype=float)

    flat_prev = states[:, :-1].reshape(-1)
    flat_next = states[:, 1:].reshape(-1)

    for i, j in zip(flat_prev, flat_next):
        counts[int(i), int(j)] += 1

    row_sums = counts.sum(axis=1, keepdims=True)
    trans = np.divide(
        counts,
        row_sums,
        out=np.zeros_like(counts),
        where=row_sums != 0
    )

    return trans


def regime_diagnostics_from_states(states, n_states):
    """
    states shape: [n_days, T]
    """
    flat = states.reshape(-1)

    usage = np.array([(flat == k).mean() for k in range(n_states)])

    effective_regimes = np.exp(
        -np.sum(usage * np.log(usage + 1e-12))
    )

    switch_rate = np.mean(states[:, 1:] != states[:, :-1])

    trans = transition_matrix_from_states(states, n_states)

    return {
        "usage": usage,
        "effective_regimes": effective_regimes,
        "switch_rate": switch_rate,
        "transition_matrix": trans,
        "transition_diag": np.diag(trans),
        "avg_transition_diag": np.diag(trans).mean(),
    }


def daily_segment_summary_from_states(states, dates):
    rows = []

    for i in range(states.shape[0]):
        s = states[i]
        switches = int(np.sum(s[1:] != s[:-1]))
        segments = switches + 1

        rows.append({
            "date": pd.to_datetime(dates[i]).date(),
            "switches": switches,
            "segments": segments,
        })

    df = pd.DataFrame(rows)

    summary = {
        "median_segments": float(df["segments"].median()),
        "mean_segments": float(df["segments"].mean()),
        "p75_segments": float(df["segments"].quantile(0.75)),
        "p90_segments": float(df["segments"].quantile(0.90)),
        "max_segments": int(df["segments"].max()),
        "one_state_share": float((df["segments"] == 1).mean()),
        "multi_state_share": float((df["segments"] > 1).mean()),
    }

    return summary, df


def regime_aware_return_diagnostic(
    val_panel,
    test_panel,
    state_col,
    return_col="ret_1",
    date_col="date",
):
    """
    Selects investable state using validation mean ret_1.
    Applies that rule to the test set.
    """
    val_state_mean_ret = (
        val_panel
        .groupby(state_col)[return_col]
        .mean()
        .sort_index()
    )

    invest_state = int(val_state_mean_ret.idxmax())

    test_tmp = test_panel.copy()
    test_tmp["invest"] = (test_tmp[state_col] == invest_state).astype(float)
    test_tmp["strategy_ret"] = test_tmp["invest"] * test_tmp[return_col]

    daily_strategy_ret = (
        test_tmp
        .groupby(date_col)["strategy_ret"]
        .sum()
    )

    daily_buy_hold_ret = (
        test_tmp
        .groupby(date_col)[return_col]
        .sum()
    )

    regime_std = daily_strategy_ret.std()
    bh_std = daily_buy_hold_ret.std()

    return {
        "invest_state": invest_state,
        "validation_state_mean_returns": val_state_mean_ret,
        "avg_daily_regime_return": float(daily_strategy_ret.mean()),
        "std_daily_regime_return": float(regime_std),
        "avg_daily_buy_hold_return": float(daily_buy_hold_ret.mean()),
        "std_daily_buy_hold_return": float(bh_std),
        "diagnostic_sharpe_regime": float(daily_strategy_ret.mean() / regime_std) if regime_std != 0 else np.nan,
        "diagnostic_sharpe_buy_hold": float(daily_buy_hold_ret.mean() / bh_std) if bh_std != 0 else np.nan,
        "annualized_diagnostic_sharpe_regime": float(np.sqrt(252) * daily_strategy_ret.mean() / regime_std) if regime_std != 0 else np.nan,
        "annualized_diagnostic_sharpe_buy_hold": float(np.sqrt(252) * daily_buy_hold_ret.mean() / bh_std) if bh_std != 0 else np.nan,
        "daily_strategy_returns": daily_strategy_ret,
        "daily_buy_hold_returns": daily_buy_hold_ret,
    }


def compute_dsssm_thesis_outputs(
    states_val,
    states_test,
    val_panel,
    test_panel,
    n_states,
    state_col,
    model_name,
    selected_key=None,
):
    """
    Computes all thesis diagnostics for one DSSSM model.
    """
    val_tmp = val_panel.copy()
    test_tmp = test_panel.copy()

    val_tmp[state_col] = states_val.reshape(-1)
    test_tmp[state_col] = states_test.reshape(-1)

    # Economic interpretation
    econ_summary = (
        test_tmp
        .groupby(state_col)[econ_cols]
        .mean()
    )

    # Future returns
    future_summary = (
        test_tmp
        .groupby(state_col)[future_cols]
        .agg(["mean", "std", "count"])
    )

    # Regime diagnostics
    diag = regime_diagnostics_from_states(states_test, n_states=n_states)

    # Daily segments
    daily_summary, daily_segments = daily_segment_summary_from_states(
        states_test,
        dates_test
    )

    # Regime-aware return
    rar = regime_aware_return_diagnostic(
        val_panel=val_tmp,
        test_panel=test_tmp,
        state_col=state_col,
        return_col="ret_1",
        date_col="date",
    )

    compact = {
        "model": model_name,
        "states": n_states,
        "selected_key": str(selected_key),
        "usage": str(np.round(diag["usage"], 6)),
        "effective_regimes": diag["effective_regimes"],
        "switch_rate": diag["switch_rate"],
        "transition_diag": str(np.round(diag["transition_diag"], 6)),
        "avg_transition_diag": diag["avg_transition_diag"],
        "median_segments": daily_summary["median_segments"],
        "mean_segments": daily_summary["mean_segments"],
        "p75_segments": daily_summary["p75_segments"],
        "p90_segments": daily_summary["p90_segments"],
        "max_segments": daily_summary["max_segments"],
        "one_state_share": daily_summary["one_state_share"],
        "invest_state": rar["invest_state"],
        "avg_daily_regime_return": rar["avg_daily_regime_return"],
        "avg_daily_buy_hold_return": rar["avg_daily_buy_hold_return"],
        "diagnostic_sharpe_regime": rar["diagnostic_sharpe_regime"],
        "diagnostic_sharpe_buy_hold": rar["diagnostic_sharpe_buy_hold"],
        "annualized_diagnostic_sharpe_regime": rar["annualized_diagnostic_sharpe_regime"],
        "annualized_diagnostic_sharpe_buy_hold": rar["annualized_diagnostic_sharpe_buy_hold"],
    }

    compact_df = pd.DataFrame([compact])

    return {
        "val_panel": val_tmp,
        "test_panel": test_tmp,
        "econ_summary": econ_summary,
        "future_summary": future_summary,
        "diagnostics": diag,
        "daily_summary": daily_summary,
        "daily_segments": daily_segments,
        "regime_aware_return": rar,
        "compact_df": compact_df,
    }


# Add future returns to panels
val_panel = add_future_returns(val_panel)
test_panel = add_future_returns(test_panel)

print("Future return columns ready:")
print(test_panel[["ret_1", "future_ret_30", "future_ret_60", "future_ret_120"]].head())

Future return columns ready:
      ret_1  future_ret_30  future_ret_60  future_ret_120
0  0.005669      -0.000062      -0.002412       -0.001731
1 -0.000618       0.000247      -0.002197       -0.001144
2  0.000000       0.000062      -0.002073       -0.001547
3  0.000772      -0.000711      -0.002598       -0.002520
4 -0.000278      -0.000371      -0.002196       -0.002939


In [36]:
# Load saved 2-state DSSSM test states
states_test_dsssm_2 = np.load(OUT_DIR / "dsssm_2state_test_states.npy")
probs_test_dsssm_2 = np.load(OUT_DIR / "dsssm_2state_test_probs.npy")

print("2-state test states:", states_test_dsssm_2.shape)
print("2-state test probs:", probs_test_dsssm_2.shape)

# Load validation states if already saved; otherwise compute them from active model
val_states_path_2 = OUT_DIR / "dsssm_2state_val_states.npy"

if val_states_path_2.exists():
    states_val_dsssm_2 = np.load(val_states_path_2)
    print("Loaded saved 2-state validation states:", states_val_dsssm_2.shape)
else:
    print("Validation states not found. Extracting using final_dsssm_model...")
    states_val_dsssm_2, probs_val_dsssm_2 = extract_dsssm_regimes(
        final_dsssm_model,
        validX,
        validY,
    )
    np.save(OUT_DIR / "dsssm_2state_val_states.npy", states_val_dsssm_2)
    np.save(OUT_DIR / "dsssm_2state_val_probs.npy", probs_val_dsssm_2)
    print("Saved 2-state validation states:", states_val_dsssm_2.shape)

# Load selected key
ckpt_2 = torch.load(OUT_DIR / "dsssm_2state_selected_model.pt", map_location=device)
best_key_2 = ckpt_2["best_key"]

# Compute all outputs
out_dsssm_2 = compute_dsssm_thesis_outputs(
    states_val=states_val_dsssm_2,
    states_test=states_test_dsssm_2,
    val_panel=val_panel,
    test_panel=test_panel,
    n_states=2,
    state_col="state_dsssm_2",
    model_name="DSSSM 2-state",
    selected_key=best_key_2,
)

display(out_dsssm_2["compact_df"])
display(out_dsssm_2["econ_summary"])
display(out_dsssm_2["future_summary"])

print("Transition matrix:")
print(np.round(out_dsssm_2["diagnostics"]["transition_matrix"], 6))

print("\nValidation mean returns by state:")
print(out_dsssm_2["regime_aware_return"]["validation_state_mean_returns"])

2-state test states: (1453, 391)
2-state test probs: (1453, 391, 2)
Loaded saved 2-state validation states: (1258, 391)


,model,states,selected_key,usage,effective_regimes,switch_rate,transition_diag,avg_transition_diag,median_segments,mean_segments,...,p90_segments,max_segments,one_state_share,invest_state,avg_daily_regime_return,avg_daily_buy_hold_return,diagnostic_sharpe_regime,diagnostic_sharpe_buy_hold,annualized_diagnostic_sharpe_regime,annualized_diagnostic_sharpe_buy_hold
0,DSSSM 2-state,2,"(2, 2, 16, 5.0, 0.001)",[0.363472 0.636528],1.925893,0.021416,[0.968442 0.984372],0.976407,9.0,9.352374,...,15.0,28,0.0,0,0.004259,0.000497,0.406367,0.0373,6.450873,0.592127


,ret_mean_30,ret_mean_60,ret_mean_120,momentum_30,momentum_60,momentum_120,trend_strength_30,trend_strength_60,trend_strength_120,ret_std_60,rv_60
state_dsssm_2,,,,,,,,,,,
0,0.000044,0.000033,0.000020,0.001297,0.001961,0.002362,0.666604,0.750927,0.645285,0.000237,0.000009
1,-0.000024,-0.000018,-0.000011,-0.000714,-0.001072,-0.001277,-0.274738,-0.311995,-0.250826,0.000348,0.000012


future_ret_30                   future_ret_60                    \
                       mean       std   count          mean       std   count   
state_dsssm_2                                                                   
0                  0.000052  0.002718  194276      0.000079  0.003704  180988   
1                 -0.000001  0.002462  331710      0.000009  0.003411  301408   

              future_ret_120                    
                        mean       std   count  
state_dsssm_2                                   
0                   0.000101  0.005054  154480  
1                   0.000044  0.004698  240736

Transition matrix:
[[0.968442 0.031558]
 [0.015628 0.984372]]

Validation mean returns by state:
state_dsssm_2
0    0.000017
1   -0.000007
Name: ret_1, dtype: float64


In [37]:
out_dsssm_2["compact_df"].to_csv(
    OUT_DIR / "summary_dsssm_2state_metrics.csv",
    index=False
)

out_dsssm_2["econ_summary"].to_csv(
    OUT_DIR / "econ_summary_dsssm_2.csv"
)

out_dsssm_2["future_summary"].to_csv(
    OUT_DIR / "future_summary_dsssm_2.csv"
)

out_dsssm_2["daily_segments"].to_csv(
    OUT_DIR / "dsssm_2state_test_daily_segments_recomputed.csv",
    index=False
)

out_dsssm_2["regime_aware_return"]["daily_strategy_returns"].to_csv(
    OUT_DIR / "dsssm_2state_daily_regime_aware_returns.csv"
)

out_dsssm_2["regime_aware_return"]["daily_buy_hold_returns"].to_csv(
    OUT_DIR / "dsssm_2state_daily_buy_hold_returns.csv"
)

out_dsssm_2["test_panel"].to_parquet(
    OUT_DIR / "test_panel_dsssm_2state.parquet",
    index=False
)

print("Saved all DSSSM 2-state thesis outputs.")

Saved all DSSSM 2-state thesis outputs.
